# Visualize structural metrics

Example of visualizing structural metrics such as distances, angles, and dihedral angles:

<video controls src="./assets/gluhut_cvs.webm">

Load a recording (unbinding of a sugar from GluHUT) as an MDAnalsis universe:

In [1]:
from MDAnalysis.lib.transformations import quaternion_about_axis, rotation_matrix

from nanover.mdanalysis import universe_from_recording

universe = universe_from_recording("../systems/recordings/gluhut-unbinding.nanover.zip")

Set up the server with playback of the universe:

In [2]:
from nanover.app import OmniRunner
from nanover.mdanalysis import UniverseSimulation

simulation = UniverseSimulation.from_universe(universe, name="nanotube + methane")
simulation.playback_factor = 30
simulation.load()

OmniRunner.close_all_runners()
imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: visualize structural metrics")
imd_runner.print_basic_info()
imd_runner.load(0)

Serving "EXAMPLE: visualize structural metrics" (ws://localhost:57825), discoverable on all interfaces on port 54545
Available simulations:
[0]: "nanotube + methane"
Switched to [0]: "nanotube + methane"
Switched to [0]: "nanotube + methane"


C:\Users\ragzo\Documents\REPOS\nanover-server-py-uv\.venv\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt


Import the jupyter utilities for drawing + interaction:

In [3]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

C:\Users\ragzo\Documents\REPOS\nanover-server-py-uv\.venv\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt


Switch to a liquorice rendering to cut down on visual noise:

In [4]:
utilities.selections.update_selection("root", renderer="liquorice")

Define a set of utility classes for extracting positional data, computing angles and distances, and representing them as lines and text:

In [5]:
import numpy as np
from typing import Sequence
from nanover.trajectory import FrameData
from MDAnalysis.lib.distances import calc_dihedrals, calc_angles


def calculate_angle_label_position(a, b, c):
    # place a little ways inside the corner of the angle
    delta = (a + c) / 2 - b
    dir = delta / np.linalg.norm(delta)
    return b + dir * .1


WHITE = [1.0, 1.0, 1.0, 1.0]
RED = [1.0, 0.0, 0.0, 1.0]
GREEN = [0.0, 1.0, 0.0, 1.0]
BLUE = [0.0, 0.5, 1.0, 1.0]


def decompose_distance(start, end):
    delta = end - start
    length = np.linalg.norm(delta)
    direction = delta / length
    return direction, length


def make_arc(*, axis, forward, angle, count=8):
    da = angle / (count - 1)
    return [forward @ rotation_matrix(i * da, axis)[:3, :3] for i in range(0, count)]


class PositionMetricVisual:
    color = WHITE

    def __init__(self, key: str, *args: Sequence[int], color=None):
        self.key = key
        self.groups = args
        self.color = color if color is not None else self.color

    def compute_positions(self, frame: FrameData):
        return [np.mean(frame.particle_positions[atoms], axis=0) for atoms in self.groups]

    def render(self, frame: FrameData):
        pass


class DihedralObject(PositionMetricVisual):
    color = BLUE

    def render(self, frame: FrameData):
        a, b, c, d = self.compute_positions(frame)
        angle = calc_dihedrals(a, b, c, d)

        radius = .15
        midpoint = (b + c) * .5

        bc, _ = decompose_distance(b, c)
        ab, _ = decompose_distance(a, b)
        axis = -bc

        # project ab onto plane normal to axis
        proj = (np.dot(ab, axis) / np.dot(axis, axis)) * axis
        ab_in_plane = ab - proj
        ab_in_plane /= np.linalg.norm(ab_in_plane)
        ab_in_plane *= -1

        # line between all points
        utilities.objects.update_line(f"{self.key}.abcd", positions=[a, b, c, d], color=self.color, size=0.01)

        # arc of dihedral angle
        arc = np.multiply([[0, 0, 0], *make_arc(axis=axis, forward=ab_in_plane, angle=angle), [0, 0, 0]], radius) + midpoint
        utilities.objects.update_line(f"{self.key}.arc", positions=arc, size=0.01, color=self.color)

        # angle label just above arc
        label = midpoint + ab_in_plane @ rotation_matrix(angle * .5, axis)[:3, :3] * (radius + .05)
        utilities.objects.update_label(
            self.key,
            position=label,
            text=f"{float(np.rad2deg(angle)):.1f}deg",
            color=self.color,
        )



class AngleObject(PositionMetricVisual):
    color = RED

    def render(self, frame: FrameData):
        a, b, c = self.compute_positions(frame)
        angle = calc_angles(a, b, c)

        # angle is between two lines ba and bc
        ba, bamag = decompose_distance(b, a)
        bc, bcmag = decompose_distance(b, c)

        # arc radius is always a little way from the end of the smallest line
        radius = min(bcmag * .75, bamag * .75, .2)

        # axis of rotation is perpendicular to the two lines
        axis = np.cross(bc, ba)

        # use a faded version of the color
        fade = [*self.color[:3], 0.1]

        # lines between points
        utilities.objects.update_line(
            f"{self.key}.abc",
            positions=[a, b+ba*radius, b, b+bc*radius, c],
            colors=[fade, self.color, self.color, self.color, fade],
            size=0.01,
        )

        # arc joining the two lines
        arc = np.multiply(make_arc(axis=axis, forward=ba, angle=angle), radius) + b
        utilities.objects.update_line(f"{self.key}.arc", positions=arc, size=0.01, color=self.color)

        # angle label just above midpoint of arc
        label = b + ba @ rotation_matrix(angle * .5, axis)[:3, :3] * (radius + .05)
        utilities.objects.update_label(
            self.key,
            position=label,
            text=f"{float(np.rad2deg(angle)):.1f}deg",
            color=self.color,
        )


class DistanceObject(PositionMetricVisual):
    color = GREEN

    def render(self, frame: FrameData):
        a, b = self.compute_positions(frame)
        distance = np.linalg.norm(b - a)

        # line between points + distance label at the midpoint
        utilities.objects.update_line(self.key, positions=[a, b], size=0.01, color=self.color)
        utilities.objects.update_label(self.key, position=(a + b) / 2, text=f"{distance:.2f}nm", color=self.color)

Define the groups of atoms of interest and the structural metrics to visualize:

In [6]:
from colorsys import hsv_to_rgb

def make_color(hue):
    return [*hsv_to_rgb(hue, .75, 1), .75]

# rings
r1_atoms = [46, 47, 50, 55, 58, 79]  # top ring
r2_atoms = [2, 3, 4, 7, 24, 27]  # bottom ring
r3_atoms = [32, 33, 34, 35, 41, 40]  # exit p3

# ligand atoms
l1_atoms = [162]
l2_atoms = [173]
l3_atoms = [159]

dihedrals = [
    DihedralObject("D1", r1_atoms, l1_atoms, l2_atoms, l3_atoms, color=make_color(0)),
    DihedralObject("D2", r2_atoms, r1_atoms, l1_atoms, l2_atoms, color=make_color(.2)),
    DihedralObject("D3", r3_atoms, r2_atoms, r1_atoms, l1_atoms, color=make_color(.4)),
]

angles = [
    AngleObject("A1", r1_atoms, l1_atoms, l2_atoms, color=make_color(0)),
    AngleObject("A2", r2_atoms, r1_atoms, l1_atoms, color=make_color(.2)),
]

distances = [
    DistanceObject("D1", l1_atoms, r1_atoms, color=make_color(0)),
]

metrics = distances

Define commands to switch between the different metrics of interest:

In [7]:
def show_distances():
    global metrics
    metrics = distances
    utilities.objects.clear()

def show_angles():
    global metrics
    metrics = angles
    utilities.objects.clear()

def show_dihedrals():
    global metrics
    metrics = dihedrals
    utilities.objects.clear()

utilities.define_command("user/distances", handler=show_distances, icon="📏", label="show distances")
utilities.define_command("user/angles", handler=show_angles, icon="🌈", label="show angles")
utilities.define_command("user/dihedrals", handler=show_dihedrals, icon="📐", label="show dihedrals")

Define and start a simple FrameListener that rerenders the active visuals when the frame is updated:

In [8]:
from nanover.jupyter import FrameListener


class MetricVisuals(FrameListener):
    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        with utilities.objects:
            for object in metrics:
                object.render(full_frame)


visuals = MetricVisuals.from_runner(imd_runner)
visuals.start()